# Section 1 — Setup & Imports

In [ ]:
# =========================
# Section 1 — Setup & Imports
# =========================
import os
import json
import re
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()

# Section 2 — Document Loading: PDFs

In [ ]:
# =========================
# Section 2 — Document Loading: PDFs
# =========================
from langchain.document_loaders import PyPDFLoader

pdf_path = "./docs/MachineLearning-Lecture01.pdf"
loader = PyPDFLoader(pdf_path)
pages = loader.load()

print("Total PDF pages:", len(pages))
print("\nPreview:\n", pages[0].page_content[:500])
print("\nMetadata:\n", pages[0].metadata)

# Section 3 — Document Loading: YouTube (Online-First with Local Fallback)

# Document Loading

In [ ]:
# =========================
# Section 3 — Document Loading: YouTube (Online-First with Local Fallback)
# =========================
from langchain_community.document_loaders import YoutubeLoader, TextLoader
from langchain.schema import Document

url = "https://www.youtube.com/watch?v=jGwO_UgTS7I"
save_dir = Path("./docs/youtube")
save_dir.mkdir(parents=True, exist_ok=True)

def extract_video_id(youtube_url: str) -> str:
    m = re.search(r"(?:v=|be/)([A-Za-z0-9_-]{11})", youtube_url)
    if not m:
        raise ValueError(f"Could not extract video id from URL: {youtube_url}")
    return m.group(1)

vid = extract_video_id(url)
txt_path = save_dir / f"{vid}.transcript.txt"
json_path = save_dir / f"{vid}.transcript.json"

def load_from_local() -> list[Document] | None:
    if txt_path.exists():
        return TextLoader(str(txt_path)).load()
    if json_path.exists():
        data = json.loads(json_path.read_text(encoding="utf-8"))
        joined = " ".join(seg.get("text", "") for seg in data)
        return [Document(page_content=joined, metadata={"source": str(json_path), "video_id": vid})]
    return None

docs = None

# 3.1 Try online fetch (works locally; may be blocked in cloud VMs)
try:
    y_loader = YoutubeLoader.from_youtube_url(url, add_video_info=False)
    docs = y_loader.load()
    # Cache a clean text transcript for future runs
    cleaned_text = docs[0].page_content if docs else ""
    txt_path.write_text(cleaned_text, encoding="utf-8")
    print("Fetched transcript online and cached to:", txt_path)
except Exception as e:
    print(f"Online transcript fetch failed ({type(e).__name__}): {e}")
    print("Attempting to load a pre-downloaded local transcript...")

# 3.2 Fallback to local cache
if docs is None:
    local_docs = load_from_local()
    if local_docs:
        docs = local_docs
        print("Loaded transcript from local cache:", txt_path if txt_path.exists() else json_path)
    else:
        raise RuntimeError(
            "Could not load YouTube transcript.\n"
            "- If running in the cloud, YouTube may block the VM.\n"
            "- Fix by running locally once (to cache) or place a transcript at:\n"
            f"  {txt_path}  (plain text)  OR  {json_path}  (list of segments)."
        )

print("\nYouTube transcript preview:\n", docs[0].page_content[:500])


## Section 4 — Document Loading: URLs (Web Pages)

In [ ]:
# =========================
# Section 4 — Document Loading: URLs (Web Pages)
# =========================
import os
os.environ['USER_AGENT'] = 'myagent'

from langchain.document_loaders import WebBaseLoader

web_url = "https://python.langchain.com/v0.2/docs/integrations/document_loaders/youtube_audio/"
w_loader = WebBaseLoader(web_url)
web_docs = w_loader.load()

print("Web page preview:\n", web_docs[0].page_content[:500].strip())

## Section 5 — Document Loading: Notion Directory

In [ ]:
# =========================
# Section 5 — Document Loading: Notion Directory
# =========================
from langchain.document_loaders import NotionDirectoryLoader

notion_dir = "./docs/Notion_DB"
n_loader = NotionDirectoryLoader(notion_dir)
notion_docs = n_loader.load()

print("Notion doc preview:\n", notion_docs[0].page_content[:200])
print("\nNotion metadata:\n", notion_docs[0].metadata)